# HackerNews Analytics Pro — v8 Explainable Intelligence

This notebook adds title-token co-occurrence networks, a transparent lexicon sentiment baseline, and holdout permutation importance for the v7 classifier. The graph is co-occurrence, not a semantic knowledge graph. Permutation importance is descriptive and can be unstable on small holdouts.

In [ ]:
from pathlib import Path
import pandas as pd
from hn_research_v8 import build_topic_network, explain_classifier, export_v8_research

DATA = Path('../demo_stories.jsonl')
df = pd.read_json(DATA, lines=True)
df.head()

## 1. Title sentiment baseline
The score is a small keyword lexicon heuristic, not a contextual transformer model.

In [ ]:
from hn_nlp import sentiment_analysis
sentiment = sentiment_analysis(df)
sentiment['sentiment_label'].value_counts()

## 2. Topic co-occurrence network
Each edge counts how often two title terms occur together in a story title.

In [ ]:
graph = build_topic_network(df, min_cooccurrence=2, max_terms=40)
print(f"{len(graph['nodes'])} nodes; {len(graph['edges'])} edges")
pd.DataFrame(graph['edges']).head(20)

## 3. Model explainability
Permutation importance shuffles one raw input column at a time on the chronological holdout and measures ROC-AUC change. Negative values can occur and do not imply a causal effect.

In [ ]:
importance = explain_classifier(df, repeats=5)
importance

## 4. Export reproducible artifacts
Run the same workflow from the terminal with `hn-analytics intelligence --input demo_stories.jsonl --output-dir reports`.

In [ ]:
summary = export_v8_research(df, output_dir='../reports', repeats=5)
summary